# QostBILLions — Soup Kernel Challenge

Final validation of the frozen implementation `5c1bbe560c08b0323eefeb843a0baf00e400e774`.

Fixed Qwen2.5-1.5B NF4 + LoRA workload on Kaggle T4. Two base and two solution runs, cold compiler caches, unmodified official harness. This notebook uses the release defaults. Validation evidence is exported to `final-evidence/`.

In [ ]:
REPO_URL="https://github.com/tokha-t/decik.git"
COMMIT="5c1bbe560c08b0323eefeb843a0baf00e400e774"


In [ ]:
import os,sys,subprocess,importlib.metadata as metadata
names=subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True,check=True).stdout
print(names)
assert names.count("T4")==2, "Select GPU T4 x2"
assert not {k:v for k,v in os.environ.items() if k.startswith("SKC_")}, "Use frozen defaults without SKC overrides"
packages=["soup-cli[train]==0.77.0","transformers==5.16.1","peft==0.20.0","trl==0.29.1","bitsandbytes==0.50.2","accelerate==1.14.0","pytest"]
subprocess.run([sys.executable,"-m","pip","install","-q",*packages],check=True)
versions={p:metadata.version(p) for p in ["torch","triton","soup-cli","transformers","peft","trl","bitsandbytes","accelerate"]}
print("VERSIONS",versions)
assert versions["torch"].startswith("2.11.0") and versions["triton"]=="3.6.0"


In [ ]:
import pathlib,shutil,json,hashlib,re
REPO=pathlib.Path("/kaggle/working/repo")
shutil.rmtree(REPO,ignore_errors=True)
subprocess.run(["git","clone","-q",REPO_URL,str(REPO)],check=True)
subprocess.run(["git","checkout","--detach","-q",COMMIT],cwd=REPO,check=True)
actual=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()
assert actual==COMMIT
print("FROZEN_COMMIT",actual)
REPORT=pathlib.Path("/kaggle/working/final-evidence");REPORT.mkdir(exist_ok=True)
manifest={"repository":REPO_URL,"commit":actual,"repeats":2,"solution_env":{},"versions":versions,"team":"QostBILLions","client_date":"2026-10-31","client_timezone":"Asia/Almaty"}
(REPORT/"manifest.json").write_text(json.dumps(manifest,indent=2))
subprocess.run([sys.executable,"task/prepare_data.py"],cwd=REPO,check=True)

def checked(command,label,env=None):
    proc=subprocess.run(command,cwd=REPO,env=env,capture_output=True,text=True)
    text=proc.stdout+"\n"+proc.stderr
    (REPORT/f"{label}.log").write_text(text)
    print(text,flush=True)
    proc.check_returncode()
    return text

gpu_log=checked([sys.executable,"-m","pytest","solution","-q"],"gpu-tests")
soup_log=checked([sys.executable,"-m","pytest","tests","-q"],"soup-preflight",dict(os.environ,SOUP_APPLY_SOLUTION="1"))
manifest["gpu_tests_passed"]=int(re.search(r"(\d+) passed",gpu_log).group(1))
manifest["soup_tests_passed"]=int(re.search(r"(\d+) passed",soup_log).group(1))
assert manifest["soup_tests_passed"]==79
(REPORT/"manifest.json").write_text(json.dumps(manifest,indent=2))


In [ ]:
subprocess.run([sys.executable,"harness/run_task.py","--repeats","2"],cwd=REPO,check=True)

In [ ]:
checked([sys.executable,"harness/check_rules.py"],"checks")
checked([sys.executable,"harness/score.py","runs/result.json"],"score")
for name in ("result.json","checks.json"):
    shutil.copy2(REPO/"runs"/name,REPORT/name)
result=json.loads((REPORT/"result.json").read_text())
checks=json.loads((REPORT/"checks.json").read_text())
assert hashlib.sha256((REPORT/"result.json").read_bytes()).hexdigest()==checks["result_sha256"]
assert len(checks["checks"])==10 and all(c["ok"] for c in checks["checks"])
manifest["result_sha256"]=checks["result_sha256"]
manifest["solution_sha256"]=checks["solution_sha256"]
manifest["summary"]=result["summary"]
(REPORT/"manifest.json").write_text(json.dumps(manifest,indent=2))
shutil.make_archive("/kaggle/working/final-evidence","zip",REPORT)
print("FINAL_RESULT",json.dumps(manifest))
print("ALL CHECKS PASSED — frozen release",COMMIT)
